# RuSearchRank Phase 1A — full MIRACL Russian BM25

This notebook is a thin Linux/Colab wrapper around `rusearchrank.cli`. It downloads the official 6.42 GiB compressed Lucene index, runs train top-100 and the official dev top-1000, evaluates the untouched dev run, then creates a separate stable dev top-100 run and the project candidate cache while streaming only the selected passages. It creates `artifacts/rusearchrank_phase1_results.zip`. Python dependencies and the extracted index need additional temporary space; the runner requires at least 30 GiB free.

The Lucene index, corpus/Hugging Face/Pyserini caches, Python environment, `.git`, and temporary work files are not included in the ZIP and must not be added to Git. The ZIP contains only candidate Parquet files, train top-100, raw dev top-1000, derived dev top-100, and the three Phase 1 audit JSON files. Run all 14 cells in order; the heavy operations are deliberately separated.

In [ ]:
# Cell 2 — fail-fast Linux, hardware, and disk gate (no downloads).
import os, platform, shutil, sys
from pathlib import Path

MIN_FREE_GIB = 30
disk = shutil.disk_usage('/content' if Path('/content').is_dir() else '.')
mem_kib = next((int(line.split()[1]) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')), 0) if Path('/proc/meminfo').is_file() else 0
environment = {
    'os': platform.platform(),
    'python': platform.python_version(),
    'cpu': platform.processor() or platform.machine(),
    'cpu_count': os.cpu_count(),
    'ram_gib': round(mem_kib / 1024**2, 2),
    'disk_free_gib': round(disk.free / 1024**3, 2),
}
print(environment)
if platform.system() != 'Linux':
    raise RuntimeError('This runner must execute on Linux (Google Colab is supported).')
if disk.free < MIN_FREE_GIB * 1024**3:
    raise RuntimeError(f'Insufficient free disk: {environment["disk_free_gib"]} GiB; at least {MIN_FREE_GIB} GiB is required before downloading the index.')

In [ ]:
# Cell 3 — clone the exact GitHub branch, or fast-forward an existing checkout.
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/OWNER/REPOSITORY.git'  # EDIT ME; never paste a token.
BRANCH = 'phase-1a'                                  # EDIT ME.
REPO_DIR = Path('/content/ru-search-rank')
ALLOW_OVERWRITE_RUNS_AND_CACHE = False  # Set True only for an intentional rerun.
if 'OWNER/REPOSITORY' in REPO_URL:
    raise ValueError('Set REPO_URL and BRANCH to the pushed Phase 1A branch.')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR, text=True, capture_output=True, check=True).stdout.strip())

In [ ]:
# Cell 4 — install and explicitly activate Java 21.
import glob, os, re, subprocess
from pathlib import Path

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-21-jdk-headless'], check=True)
java_candidates = sorted(glob.glob('/usr/lib/jvm/java-21*/bin/java'))
if not java_candidates:
    raise RuntimeError('OpenJDK 21 was installed but its java executable was not found.')
java_bin = Path(java_candidates[0]).resolve()
JAVA_HOME = java_bin.parent.parent
os.environ['JAVA_HOME'] = str(JAVA_HOME)
os.environ['PATH'] = f'{java_bin.parent}:' + os.environ['PATH']
java_version = subprocess.run([str(java_bin), '-version'], text=True, capture_output=True, check=True).stderr
print(java_version)
if not re.search(r'(?:version\s+"|openjdk\s+)21(?:[."]|$)', java_version.lower()):
    raise RuntimeError(f'Active Java is not 21; JAVA_HOME={JAVA_HOME}')

In [ ]:
# Cell 5 — create an isolated Python 3.12 environment without replacing system Python.
import shutil, subprocess, sys
from pathlib import Path

VENV_DIR = Path('/content/rusearchrank-py312')
if sys.version_info[:2] == (3, 12):
    python312 = Path(sys.executable)
    if not (VENV_DIR / 'bin/python').is_file():
        subprocess.run([str(python312), '-m', 'venv', str(VENV_DIR)], check=True)
else:
    UV_VERSION = '0.8.13'
    uv_prefix = Path('/content/uv-bootstrap')
    uv = uv_prefix / 'bin/uv'
    if not uv.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--prefix', str(uv_prefix), f'uv=={UV_VERSION}'], check=True)
    subprocess.run([str(uv), 'python', 'install', '3.12'], check=True)
    if not (VENV_DIR / 'bin/python').is_file():
        subprocess.run([str(uv), 'venv', '--python', '3.12', str(VENV_DIR)], check=True)
RUN_PYTHON = VENV_DIR / 'bin/python'
actual_python = subprocess.run([str(RUN_PYTHON), '--version'], text=True, capture_output=True, check=True).stdout.strip()
print(actual_python, RUN_PYTHON)
if not actual_python.startswith('Python 3.12.'):
    raise RuntimeError(f'Isolated interpreter is not Python 3.12: {actual_python}')

In [ ]:
# Cell 6 — install the project and pinned retrieval dependency, then test under Python 3.12.
import os, subprocess

subprocess.run([str(RUN_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([str(RUN_PYTHON), '-m', 'pip', 'install', '-e', f'{REPO_DIR}[retrieval]'], check=True)
version_probe = "import importlib.metadata as m,sys; print(sys.version); print('pyserini',m.version('pyserini'))"
subprocess.run([str(RUN_PYTHON), '-c', version_probe], check=True, env=os.environ.copy())
# Exact isolated-environment equivalent of: python -m pytest -q
subprocess.run([str(RUN_PYTHON), '-m', 'pytest', '-q'], cwd=REPO_DIR, env=os.environ.copy(), check=True)

In [ ]:
# Cell 7 — download official topics/qrels, then check Pyserini/Java and the index.
import os, subprocess

CONFIG = 'configs/retrieval.yaml'
def cli(*arguments):
    command = [str(RUN_PYTHON), '-m', 'rusearchrank.cli', *arguments]
    print('+', ' '.join(command))
    return subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), text=True, check=True)

cli('prepare-annotations', '--config', CONFIG)
cli('inspect-linux-environment', '--config', CONFIG, '--check-index')

In [ ]:
# Cell 8 — train BM25 top-100 from the validated local official TSV (4,683 queries).
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('run-bm25', '--config', CONFIG, '--split', 'train', *overwrite)

In [ ]:
# Cell 9 — official dev BM25 with --hits 1000; preserve the raw run for reproduction.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('run-bm25', '--config', CONFIG, '--split', 'dev', *overwrite)

In [ ]:
# Cell 10 — evaluate the untouched dev top-1000 with official trec_eval commands.
# subprocess check=True stops the notebook here unless the reproduction gate passes.
cli('evaluate-bm25', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 11 — after the gate: separate stable dev top-100 run, three-state cache, candidate passages.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('build-candidate-cache', '--config', CONFIG, *overwrite)
cli('audit-qrels', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 12 — candidate, query, passage, three-state, top-K, and stable-order validation.
cli('validate-candidates', 'artifacts/candidates/train_top100.parquet', '--config', CONFIG)
cli('validate-candidates', 'artifacts/candidates/dev_top100.parquet', '--config', CONFIG)

In [ ]:
# Cell 13 — package the explicit portable allowlist; caches/index/environment are excluded.
cli('package-phase1', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 14 — show ZIP size, SHA-256, exact contents, then download (optional Drive copy).
import hashlib, shutil, zipfile
from pathlib import Path

archive_path = REPO_DIR / 'artifacts/rusearchrank_phase1_results.zip'
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with zipfile.ZipFile(archive_path) as archive:
    contents = archive.namelist()
print({'path': str(archive_path), 'size_bytes': archive_path.stat().st_size, 'sha256': digest, 'contents': contents})

DRIVE_DESTINATION = ''  # Optional, e.g. '/content/drive/MyDrive/rusearchrank_phase1_results.zip'.
if DRIVE_DESTINATION:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy2(archive_path, DRIVE_DESTINATION)
    print('Copied to', DRIVE_DESTINATION)
from google.colab import files
files.download(str(archive_path))